In [1]:
import sopy as sp
import pandas as pd
import sopy.tensorly as stl
import tensorflow as tf
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sopy.pyscf.ext import *
import time

In [2]:
import numpy as np
from scipy.special import expi
import matplotlib.pyplot as plt
def calculate_psi(lattice, p, y):
    """
    Computes the value of the provided Mathematica expression.
    
    Parameters:
    L, d, p, y : float or complex
        Input variables corresponding to the Mathematica formula.
        
    Returns:
    complex
        The result of the calculation.
    """
    d = (lattice[1]-lattice[0])
    L = lattice[-1]-lattice[0]+d

    # Constants
    # In Python, '1j' is the imaginary unit (equivalent to Mathematica's 'I')
    I = 1j 
    Pi = np.pi
    
    # ---------------------------------------------------------
    # 1. Translate the Prefactor
    # Mathematica: -(1/(2 Sqrt[L] \[Pi])) I Sqrt[d] (E^(-I p y))
    # ---------------------------------------------------------
    
    # Calculate square roots once for clarity
    sqrt_L = np.sqrt(L)
    sqrt_d = np.sqrt(d)
    
    # Note: parentheses in the denominator (2 * sqrt_L * Pi) are crucial
    term_prefactor = -(1 / (2 * sqrt_L * Pi)) * I * sqrt_d * np.exp(-I * p * y)

    # ---------------------------------------------------------
    # 2. Translate the ExpIntegralEi terms
    # ---------------------------------------------------------
    
    # Term A: ExpIntegralEi[(I (-d p + \[Pi]) (L - 2 y))/(2 d)]
    arg_a = (I * (-d * p + Pi) * (L - 2 * y)) / (2 * d)
    term_a = expi(arg_a)
    
    # Term B: ExpIntegralEi[(I (d p + \[Pi]) (-(L/2) + y))/d]
    arg_b = (I * (d * p + Pi) * (-(L / 2) + y)) / d
    term_b = expi(arg_b)
    
    # Term C: ExpIntegralEi[(I (d p - \[Pi]) (L + 2 y))/(2 d)]
    arg_c = (I * (d * p - Pi) * (L + 2 * y)) / (2 * d)
    term_c = expi(arg_c)
    
    # Term D: ExpIntegralEi[(I (d p + \[Pi]) (L + 2 y))/(2 d)]
    arg_d = (I * (d * p + Pi) * (L + 2 * y)) / (2 * d)
    term_d = expi(arg_d)
    
    # ---------------------------------------------------------
    # 3. Combine them
    # Mathematica: (TermA - TermB - TermC + TermD)
    # ---------------------------------------------------------
    
    # 1. Construct the array with signs applied
    terms = np.array([term_a, -term_b, -term_c, term_d])
    
    # 2. Multiply by prefactor and replace any NaNs with 0.0
    # This is much faster and cleaner than list(map(...))
    result = term_prefactor *np.nan_to_num( terms, nan=0.0,posinf=0.0, 
    neginf=0.0)    
    return result

def def_crystal(lattice):
    d = lattice[1]-lattice[0]
    L = (lattice[-1]-lattice[0]+d)
    dp = 2.*np.pi/ L
    N  = len(lattice)
    N12 = N//2-0.5
    return [ dp*(n -N12) for n in range(N) ]

def ft_matrix(crystal,  lattice ):
    ft = np.transpose([[ calculate_psi(lattice, p , y) for p in crystal ] for y in lattice],(2,0,1))
    norm = 1./np.sqrt(np.mean( np.diag( np.dot(np.conj(np.transpose( sum(ft))), sum(ft)) ) ) ) 
    return norm * ft


In [3]:
def rotate_geometry(geometry_str, roll, pitch, yaw):
    # 1. Parse the string into lines and filter out empty ones
    lines = [line.strip() for line in geometry_str.strip().split('\n')]
    
    rotated_lines = []
    
    for line in lines:
        # Split by comma: [AtomName, X, Y, Z]
        parts = line.split(',')
        label = parts[0]
        # Convert coordinates to a 3D vector
        coords = np.array([float(p) for p in parts[1:]])
        # 2. "Act" with your rotator
        # Assuming your function returns the rotated vector
        new_coords = rotation_matrix_from_euler(roll, pitch, yaw, coords)
        # 3. Format back into a string with 3 decimal precision
        new_line = f"{label},{new_coords}"
        rotated_lines.append(new_line)
    return "\n".join(rotated_lines)

# Usage
# rotated_benzene = rotate_geometry(atom_geometry0, 0.1, 0.5, -0.2)
# print(rotated_benzene)

In [4]:
Data = []

In [5]:
DownSampling = 1
partition_orig = 10
tol = 1e-12
# Vector
sampling_rate = 4
# spatial setup
padding = 4

# compute extra
padding_factor = 1
roll = 0.1 
pitch = 1
yaw = -1

roll=0
pitch=0
yaw = 0


atom_geometry = rotate_geometry('''C1,1.399,0.000,0.000
C2,0.699,1.212,0.000
C3,-0.699,1.212,0.000
C4,-1.399,0.000,0.000
C5,-0.699,-1.212,0.000
C6,0.699,-1.212,0.000
H1,2.483,0.000,0.000
H2,1.241,2.150,0.000
H3,-1.241,2.150,0.000
H4,-2.483,0.000,0.000
H5,-1.241,-2.150,0.000
H6,1.241,-2.150,0.000  ''', roll,pitch,yaw)


# atom_geometry = rotate_geometry('''
# H ,-1, 0, 0 
# H ,1, 0, 0
# ''', roll,pitch,yaw)

atom_geometry


'C1, 1.399 0.0 0.0 \nC2, 0.699 1.212 0.0 \nC3, -0.699 1.212 0.0 \nC4, -1.399 0.0 0.0 \nC5, -0.699 -1.212 0.0 \nC6, 0.699 -1.212 0.0 \nH1, 2.483 0.0 0.0 \nH2, 1.241 2.15 0.0 \nH3, -1.241 2.15 0.0 \nH4, -2.483 0.0 0.0 \nH5, -1.241 -2.15 0.0 \nH6, 1.241 -2.15 0.0 '

In [6]:


from pyscf import gto, scf

def get_molecular_orbital_image(atom_str, basis, padding=3.0, sampling_rate=sampling_rate, canons = 4):
    """
    Calculates the HOMO orbital and slices it into a 2D image.
    """
    # 1. Build Molecule & Run SCF
    mol = gto.M(atom=atom_str, basis=basis, verbose=0)
    mf = scf.RHF(mol).run()
    
    # 2. Define the 2D Grid (Slice at Z=0)
    # We define a square window around the molecule
    L = np.max(np.abs(mol.atom_coords()))* padding
    grid_points = int( sampling_rate * 2 *L )
    x = np.linspace(-L, L, grid_points )
    y = np.linspace(-L, L, grid_points )
    z = np.linspace(-L, L, grid_points )
    X, Y, Z = np.meshgrid(x, y, z)
    
    coords = np.vstack([ X.flatten(), Y.flatten(),Z.flatten() ]).T

    # 3. Evaluate Orbitals on Grid
    # 'GTOval' calculates the value of the Atomic Orbitals (AO) at these points
    ao_value = mol.eval_gto("GTOval", coords)
    
    # Transform AO to Molecular Orbitals (MO) using the calculated coefficients
    # shape: (num_points, num_orbitals)
    mo_value = ao_value @ mf.mo_coeff
    
    # 4. Extract HOMO (Highest Occupied Molecular Orbital)
    # For a neutral closed shell, HOMO is at index n_elec/2 - 1
    n_occ = mol.nelectron // 2
    homo_idx = n_occ - 1
    homo_flat = mo_value[:, homo_idx]
    lattices = (x,y,z)

    # Reshape back to 2D image
    orbital_image = homo_flat.reshape(grid_points, grid_points, grid_points)

    lattices_dict= { d + 1 : lattices[d] for d in range(3)}
    
    #s = get_orbital( mol, mf.mo_coeff, homo_idx, lattices  ) 

    #s2 = sp.Vector().transpose( { 0 : (s[0]) , 1 : s[2][:,::DownSampling] , 2 : (s[1][:,::DownSampling]), 3:  s[3][:,::DownSampling] } 
                               # , lattices_dict)
    #norm = s2.n()
    #s2 = s2.copy(True).fibonacci(partition)
    return grid_points, np.array([x, y, z]), X, Y, orbital_image#, norm

def multiply_sopy( crystals, lattices, tss, s):
    """
    Computes the FFT magnitude and corresponding frequency axis.
    """

    #tss # ( canon, space , .. )
    norm = tf.sqrt(s.re.n()**2+ s.im.n()**2)
    dict_crystals = { d+1: crystal  for d,crystal in enumerate(crystals)}
    t = s.transform(dict_crystals,  tss )
    return np.array(crystals), norm, t




# --- Run the Generator ---
grid_points, lattices, X, Y, signal = get_molecular_orbital_image(atom_geometry, 'sto-3g', padding=padding, sampling_rate=sampling_rate)

In [7]:
# reset 
data = {}
lattices = [ lattice[::DownSampling] for lattice in lattices ] 
dict_lattices = { d+1 : lattice for d,lattice in enumerate(lattices) }
crystals = [ np.array(def_crystal( lattice ) ) for lattice in lattices ]
s = stl.reduce(signal[::DownSampling,::DownSampling], partition=16, tol = tol)
sopy_signal =  sp.Vector().transpose( s, dict_lattices)

In [12]:
sopy_signal = sopy_signal.max(1)

In [13]:
# iterate
Data = []
for iteration in range(6):
    data = {}
    data['lattices'] = lattices
    data['crystals'] = crystals
    data['delta'] = lattices[0][1]-lattices[0][0]
    data['max_momentum'] = max(crystals[0])
    data['length']    = len(lattices[0])
    
    start_time = time.time()
    fts = [ ft_matrix( crystal, lattice ) for crystal, lattice in zip(crystals, lattices) ]
    tss = [ { d+1 : ts[d] for d in range(len(lattices)) } for ts in list(itertools.product(*fts)) ]
    ttss = [ { d+1 : np.conj(np.transpose(ts[d])) for d in range(len(lattices)) } for ts in list(itertools.product(*fts)) ]
    end_time = time.time()
    data['build_duration'] = end_time - start_time
    
    
    start_time = time.time()
    sopy_ops    = sp.Operand( sopy_signal, sp.Vector() )
    for _ in range(1):
        (wavenumber_axis1, wavenumber_axis2, wavenumber_axis3), norm, cft_mag = multiply_sopy(crystals, lattices,  tss, sopy_ops)
    # ----------------------
    end_time = time.time()
    data['multiply_duration'] = end_time - start_time
    data['padding'] = padding
    data['sampling_rate'] = sampling_rate
    data['pitch']= pitch
    data['roll'] = roll
    data['yaw']  = yaw
    for partition in [4]:
        for partition2 in [1]:
            for alpha in [1e-6]:
                for total_iterate in [1]:
                    for iterate in [5]:
                        data['partition'] = partition
                        data['partition2'] = partition2
                        data['iterate'] = iterate
                        data['total_iterate'] = total_iterate
                        data['alpha'] = alpha
                        start_time = time.time()
                        re_mag, im_mag = cft_mag.re.n(), cft_mag.im.n()
                        if True:
                            if False:
                                o2 =  sp.Operand( 
                                              sp.Vector() if re_mag <= 1e-9 else cft_mag.re.fibonacci(
                                partition, iterate=iterate, total_iterate=total_iterate, alpha=alpha*re_mag, total_alpha=alpha*re_mag), 
                                              sp.Vector() if re_mag <= 1e-9 else cft_mag.im.fibonacci(
                                partition, iterate=iterate, total_iterate=total_iterate, alpha=alpha*im_mag, total_alpha=alpha*im_mag) 
                                            )   
                            else:
                                o2 =  sp.Operand( 
                                              sp.Vector() if re_mag <= 1e-9 else cft_mag.re.decompose(
                                partition, iterate=iterate, alpha=alpha*re_mag), 
                                              sp.Vector() if re_mag <= 1e-9 else cft_mag.im.decompose(
                                partition, iterate=iterate, alpha=alpha*im_mag) 
                                            )   

                        
                        else:
                            o2 = cft_mag
                            
                        data[f'cft_dist'] = np.real(2 - (o2.cdot(cft_mag)+cft_mag.cdot(o2))/np.sqrt( np.abs(o2.cdot(o2) * cft_mag.cdot( cft_mag) )))
                        end_time = time.time()
                        data[f'cft_decompose'] = end_time - start_time

                        start_time = time.time()
                        for _ in range(1):
                            (_, _, _), _, rendered_image = multiply_sopy( lattices, crystals,  ttss, o2)
                        # ----------------------
                        end_time = time.time()
                        data['render_duration'] = end_time - start_time
                        
                        start_time = time.time()
                        re_mag, im_mag = rendered_image.re.n(), rendered_image.im.n()
                        if False:
                            i2 =  sp.Operand( 
                                          sp.Vector() if re_mag <= 1e-9 else rendered_image.re.fibonacci(
                            partition2, iterate=iterate, total_iterate=total_iterate, alpha=alpha*re_mag, total_alpha=alpha*re_mag), 
                                          sp.Vector() if im_mag <= 1e-9 else rendered_image.im.fibonacci(
                            partition2, iterate=iterate, total_iterate=total_iterate, alpha=alpha*im_mag, total_alpha=alpha*im_mag) 
                                        )  
                        else:
                            i2 =  sp.Operand( 
                                          sp.Vector() if re_mag <= 1e-9 else rendered_image.re.decompose(
                            partition2, iterate=iterate, alpha=alpha*re_mag), 
                                          sp.Vector() if im_mag <= 1e-9 else rendered_image.im.decompose(
                            partition2, iterate=iterate, alpha=alpha*im_mag) 
                                        )    

                        data[f'return_dist'] = np.real(2 - (i2.cdot(sopy_ops)+sopy_ops.cdot(i2))/np.sqrt( np.abs(i2.cdot(i2) * sopy_ops.cdot(sopy_ops) )))
                        data[f'render_dist'] = np.real(2 - (rendered_image.cdot(sopy_ops)+sopy_ops.cdot(rendered_image))/np.sqrt( np.abs(rendered_image.cdot(rendered_image) * sopy_ops.cdot(sopy_ops) )))
                        end_time = time.time()
                        data[f'render_decompose'] = end_time - start_time

                        Data += [pd.Series(data)]    
                        pd.DataFrame(Data).to_csv(f'flatten_one_timing.csv',index = False)

    # if iteration < 3:
    #    cft_image = tf.abs(stl.image(o2.re) + 1.j * stl.image(o2.im))
    #    cft_image /= np.sum(cft_image)
    # pd.Series( np.ravel(cft_image)).to_csv(f'flatten_benenze_image_{iteration}.csv',index = False)
    # np.save(f'flatten_benenze_sop_{iteration}.npy',[ sopy_signal[space].numpy for space in o2.re.dims(False) ])

    if True:
        lattices2 = [ np.linspace(lattice[0]-1, lattice[-1]+1,len(lattice)*3//2) for lattice in lattices ] 
        sopy_signal2 = sopy_signal.resample( lattices2=lattices2, partition=len(sopy_signal) )#.fibonacci(1, iterate =10) 
        sopy_signal = sopy_signal2
        lattices    = lattices2
        crystals = [ np.array(def_crystal( lattice ) ) for lattice in lattices ]


In [14]:
pd.DataFrame( Data)

,lattices,crystals,delta,max_momentum,length,build_duration,multiply_duration,padding,sampling_rate,pitch,...,partition2,iterate,total_iterate,alpha,cft_dist,cft_decompose,render_duration,return_dist,render_dist,render_decompose
0,"[[-18.768759869180194, -18.516830206506633, -1...","[[-12.386984007554014, -12.220715765841879, -1...",0.251930,12.386984,150,1.185729,0.291759,4,4,0,...,1,5,1,0.000001,6.661338e-16,0.076162,0.269565,7.656098e-13,7.653878e-13,0.077554
1,"[[-19.768759869180194, -19.592253084633942, -1...","[[-17.640496899664388, -17.48228616513825, -17...",0.176507,17.798708,225,2.699961,1.113600,4,4,0,...,1,5,1,0.000001,8.881784e-16,1.083176,4.390596,9.828360e-11,9.828316e-11,0.494774
2,"[[-20.768759869180194, -20.64513629853031, -20...","[[-25.26175364767211, -25.110937207984513, -24...",0.123624,25.412570,337,6.092845,2.506261,4,4,0,...,1,5,1,0.000001,-4.440892e-16,2.345904,9.429795,6.182754e-10,6.182743e-10,1.092215
3,"[[-21.768759869180194, -21.68237590144535, -21...","[[-36.22374364756526, -36.07971285771411, -35....",0.086384,36.367774,505,13.846244,5.648637,4,4,0,...,1,5,1,0.000001,-4.440892e-16,5.314757,21.249545,8.747416e-10,8.747412e-10,2.523767
4,"[[-22.768759869180194, -22.708525054711462, -2...","[[-52.017966163492346, -51.880170226635414, -5...",0.060235,52.155762,757,30.658358,13.903487,4,4,0,...,1,5,1,0.000001,-4.440892e-16,12.325642,48.924201,9.446139e-10,9.446155e-10,5.758586
5,"[[-23.768759869180194, -23.726839657770704, -2...","[[-74.8101381662953, -74.67808143519858, -74.5...",0.041920,74.942195,1135,70.876153,30.746279,4,4,0,...,1,5,1,0.000001,-4.440892e-16,28.512100,134.411712,1.422714e-09,1.422710e-09,27.095345
